In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from torch.cuda.amp import GradScaler, autocast
import os
import time  # Added for timing information

# Define your model save path
MODEL_SAVE_PATH = "./motherland_savior_model_12_3_P.pth"

class DragonWarriorModel(nn.Module):
    def __init__(self, num_classes):
        print("\n starting model construction")
        super().__init__()
        base = models.resnet50(pretrained=True)
        print("🔄 Loading ResNet50 base model...")
        
        for param in base.parameters():
            param.requires_grad = False
        print("freezing base model parameters")
            
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),  # ADDED FOR STABILITY
            nn.ReLU(),
            nn.Dropout(0.3),  # OPTIMIZED FOR SPEED
            nn.Linear(1024, num_classes)
        )
        print(f" Classifier constructed with {num_classes} output classes")

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

def execute_shock_doctrine(dataset):
    print("\n training sequence initiated")
    
    # Print dataset info for debugging
    print(f"📊 Dataset contains {len(dataset)} samples across {len(dataset.classes)} classes")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ Using device: {device}")
    
    if torch.cuda.is_available():
        print(f"🎮 GPU count: {torch.cuda.device_count()}")
        print(f"🎮 GPU name: {torch.cuda.get_device_name(0)}")
    
    # MULTI-GPU DEPLOYMENT
    model = DragonWarriorModel(len(dataset.classes))
    if torch.cuda.device_count() > 1:
        print(f"⚡ DEPLOYING TO {torch.cuda.device_count()} T4 WAR MACHINES ⚡")
        model = nn.DataParallel(model)
    model = model.to(device)

    print("Model deployed to GPU(s)")

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    
    # DATA LOADING
    dataloader = DataLoader(
        dataset,
        batch_size=128,  # DOUBLE THE FIREPOWER
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )
    print(f"📦 Created DataLoader with {len(dataloader)} batches")
    
    # TRAINING SEQUENCE
    start_time = time.time()
    for epoch in range(15): #CHANGE EPOCH HERE
        epoch_start = time.time()
        print(f"\n⚔️ EPOCH {epoch+1}/{15} STARTED")
        model.train()
        total_loss = 0
        batch_count = 0
        
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            if batch_idx % 10 == 0:
                print(f"  ⏳ Processing batch {batch_idx}/{len(dataloader)} ({batch_idx/len(dataloader)*100:.1f}%)")
            
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)  # MEMORY OPTIMIZATION
            
            total_loss += loss.item()
            batch_count += 1
        
        epoch_time = time.time() - epoch_start
        print(f"COMBINED LOSS: {total_loss/len(dataloader):.4f} | ⏱️ Epoch time: {epoch_time:.2f}s")
        
    total_time = time.time() - start_time
    print(f"TOTAL TRAINING TIME: {total_time:.2f}s ({total_time/60:.2f}min)")
    
    print(f"Saving model to {MODEL_SAVE_PATH}...")
    torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), 
             MODEL_SAVE_PATH)
    print(f"\n model deployed at.. {MODEL_SAVE_PATH} 🛡️")

if __name__ == "__main__":
    print("initiating shock doctrine sequence")
    
    # Use your data_vault from the previous cell
    if 'data_vault' in globals():
        print(f"Found data_vault variable")
        execute_shock_doctrine(data_vault)
    else:
        print("ERROR: data_vault not found! Run your data preparation cell first.")
        print("TIP: Make sure you've defined 'data_vault' in a previous cell")


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image

# --- CONFIG ---
MODEL_PATH = '//kaggle/input/dragon/pytorch/default/1/motherland_savior_model_12_3_P.pth'
TEST_DIR = '/kaggle/input/tammathon-task-1/test/test'
CSV_PATH = '/kaggle/input/tammathon-task-1/test.csv'
SUBMIT_PATH = '/kaggle/working/submission_10_P3.csv'
NUM_CLASSES = 113592  # Change if needed

# --- PREPROCESSING ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- MODEL LOADING ---
class DragonWarriorModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.resnet50(weights=None)
        for param in base.parameters():
            param.requires_grad = False
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DragonWarriorModel(NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
model.to(device)

# --- PREDICTION ---
df = pd.read_csv(CSV_PATH)
predictions = []

for idx, row in df.iterrows():
    img_path = f"{TEST_DIR}/{row['filename'].split('/')[-1]}"
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_tensor)
        top3 = torch.topk(logits, 3).indices.cpu().numpy().flatten()
    
    predictions.append([row['filename']] + top3.tolist())

# --- SUBMISSION ---
submission_df = pd.DataFrame(predictions, columns=['filename', 'label_1', 'label_2', 'label_3'])
submission_df.to_csv(SUBMIT_PATH, index=False)
print(f"✅ Submission file saved to: {SUBMIT_PATH}")
